# NIfTI, resampling, and registration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cest-sources/imaging_science_tools/blob/cursor/merge-nifti-registration-7d5a/4_Image_visualization_and_registration/nifti_and_registration.ipynb)

A scan arrives as DICOM: many files, one per slice. The usual step is to convert one series into one NIfTI. After that, this notebook stays in NIfTI.

A NIfTI is two arrays.

- **Data array.** Intensities at voxel indices `(i, j, k)`.
- **Voxel-to-world matrix.** A 4×4 affine. An index goes in, a point in millimetres comes out. The length of a column is the voxel size. The direction of a column is the tilt. The last column is the origin of voxel `(0, 0, 0)`.

That split is why the format is convenient. Resampling, overlay, and registration all use the same pair: the array says what a voxel holds, the affine says where it sits.

`dcm2niix` does the conversion. In Python, SimpleITK does the same for one series:

```python
import SimpleITK as sitk

def dicom_series_to_nifti(dicom_dir, nifti_path):
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(reader.GetGDCMSeriesFileNames(dicom_dir))
    sitk.WriteImage(reader.Execute(), nifti_path)
```

The examples below start from NIfTI. First a contrast MRI is resliced onto a tilted slab and back. Then that high-resolution scan is registered onto the low-resolution slab, once from its affine and once by a search (ANTs, then DIPY).

Every picture is NiiVue. It reads the affine. Buttons switch radiologic and neurologic view, and world millimetres and voxel indices. With two images, a checkbox turns each one on or off. Slice edges are labeled R, L, A, P, S, and I (S is head, I is foot). On GitHub the panels are empty until you run the cells.


In [1]:
!pip install -q nibabel scipy ipywidgets ipyniivue antspyx dipy


In [2]:
from pathlib import Path
import urllib.request

import ipywidgets as widgets
import nibabel as nib
import numpy as np
from nibabel.affines import apply_affine
from ipyniivue import MultiplanarType, NiiVue, ShowRender, SliceType, Volume

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

DATA = Path("data")
OUT = Path("output")
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

MR_GD_URL = "https://niivue.github.io/niivue-demo-images/MR_Gd.nii.gz"

def fetch(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def save(data, affine, name):
    path = OUT / name
    nib.save(nib.Nifti1Image(np.asarray(data, np.float32), affine), path)
    return path

def rotation_about_x(degrees):
    theta = np.deg2rad(degrees)
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1.0, 0.0, 0.0],
                      [0.0, c, -s],
                      [0.0, s, c]])

def directions_of(affine):
    # Unit directions of the voxel axes, as columns.
    linear = np.asarray(affine[:3, :3], dtype=float)
    spacing = np.linalg.norm(linear, axis=0)
    return linear / spacing

def grid_affine(shape, spacing, rotation, center_world):
    # 4x4 affine. The center of the grid sits at center_world, in mm.
    shape = np.asarray(shape, dtype=float)
    spacing = np.asarray(spacing, dtype=float)
    center_index = (shape - 1) / 2.0
    origin = np.asarray(center_world, dtype=float) - rotation @ (spacing * center_index)
    affine = np.eye(4)
    affine[:3, :3] = rotation @ np.diag(spacing)
    affine[:3, 3] = origin
    return affine

def resample_to_grid(source, source_affine, target_shape, target_affine, order=1, mode="constant"):
    # For each target voxel: index -> world mm -> source index, then interpolate.
    # mode="constant" writes 0 outside the source. mode="nearest" keeps the edge voxel,
    # which matters when a sample lands on the last index of a thin slab.
    axes = [np.arange(n) for n in target_shape]
    grids = np.meshgrid(*axes, indexing="ij")
    target_ijk = np.stack(grids, axis=-1)
    world = apply_affine(target_affine, target_ijk)
    source_ijk = apply_affine(np.linalg.inv(source_affine), world)
    coords = np.moveaxis(source_ijk, -1, 0)
    sampled = map_coordinates_safe(source, coords, order=order, mode=mode)
    return sampled

def map_coordinates_safe(source, coords, order, mode="constant"):
    from scipy.ndimage import map_coordinates
    sampled = map_coordinates(source, coords, order=order, mode=mode, cval=0.0)
    return sampled.astype(np.float32)

def resample_footprint(source, source_affine, target_shape, target_affine, samples_per_axis):
    # Average several samples inside each target voxel, not only the center.
    # samples_per_axis is (ni, nj, nk) inside the voxel cube [-0.5, 0.5].
    from scipy.ndimage import map_coordinates
    offsets = [(np.arange(n) + 0.5) / n - 0.5 for n in samples_per_axis]
    ii, jj, kk = np.meshgrid(*[np.arange(n) for n in target_shape], indexing="ij")
    base = np.stack([ii, jj, kk], axis=-1).astype(float)
    acc = np.zeros(target_shape, dtype=np.float64)
    count = 0
    for di in offsets[0]:
        for dj in offsets[1]:
            for dk in offsets[2]:
                target_ijk = base + np.array([di, dj, dk])
                world = apply_affine(target_affine, target_ijk)
                source_ijk = apply_affine(np.linalg.inv(source_affine), world)
                coords = np.moveaxis(source_ijk, -1, 0)
                acc += map_coordinates(source, coords, order=1, mode="constant", cval=0.0)
                count += 1
    return (acc / count).astype(np.float32)

def show_niivue(paths, colormaps=None, opacities=None, vmax=None, height=560):
    paths = list(paths)
    if colormaps is None:
        colormaps = ["gray"] * len(paths)
    if opacities is None:
        opacities = [1.0] * len(paths)
    volumes = []
    for path, cmap, opacity in zip(paths, colormaps, opacities):
        kw = dict(path=path, colormap=cmap, opacity=opacity, cal_min=0.0)
        if vmax is not None:
            kw["cal_max"] = float(vmax)
        volumes.append(Volume(**kw))
    nv = NiiVue(
        height=height,
        slice_type=SliceType.MULTIPLANAR,
        multiplanar_show_render=ShowRender.ALWAYS,
        multiplanar_layout=MultiplanarType.GRID,
        is_slice_mm=True,
        is_colorbar=True,
        is_radiological_convention=False,
        is_orientation_text_visible=True,
        show_all_orientation_markers=True,
        is_corner_orientation_text=False,
        font_color=(1.0, 1.0, 1.0, 1.0),
        font_min_px=16,
    )
    nv.load_volumes(volumes)

    view_button = widgets.Button(
        description="view: neurologic",
        layout=widgets.Layout(width="220px"),
    )
    coord_button = widgets.Button(
        description="coordinates: world mm",
        layout=widgets.Layout(width="220px"),
    )

    def flip_view(_):
        radiological = not bool(nv.opts.is_radiological_convention)
        nv.set_radiological_convention(radiological)
        view_button.description = (
            "view: radiologic" if radiological else "view: neurologic"
        )

    def flip_coords(_):
        world = not bool(nv.opts.is_slice_mm)
        nv.set_slice_mm(world)
        coord_button.description = (
            "coordinates: world mm" if world else "coordinates: voxel"
        )

    view_button.on_click(flip_view)
    coord_button.on_click(flip_coords)

    rows = [widgets.HBox([view_button, coord_button])]
    if len(paths) > 1:
        checks = []
        for index, (path, opacity) in enumerate(zip(paths, opacities)):
            label = Path(path).name.removesuffix(".nii.gz").removesuffix(".nii")
            check = widgets.Checkbox(
                value=True,
                description=label,
                indent=False,
                layout=widgets.Layout(width="max-content"),
            )

            def on_image(change, vol_idx=index, base_opacity=float(opacity)):
                nv.set_opacity(vol_idx, base_opacity if change["new"] else 0.0)

            check.observe(on_image, names="value")
            checks.append(check)
        rows.append(widgets.HBox(checks))

    note = widgets.HTML(
        "Slice edges: <b>R</b> <b>L</b> right and left, "
        "<b>A</b> <b>P</b> anterior and posterior, "
        "<b>S</b> <b>I</b> superior and inferior (head and foot)."
    )
    rows.append(note)
    rows.append(nv)
    return widgets.VBox(rows)


## Load a NIfTI

`MR_Gd.nii.gz` is a skull-stripped contrast MRI, about 1 mm. The niivue demo set has no separate `MR_Gd_highres.nii.gz`, so this file is the high-resolution volume.

The printout is the data array (`shape`) and the voxel-to-world matrix (`affine`).


In [3]:
path_high = fetch(MR_GD_URL, "MR_Gd.nii.gz")
high_img = nib.load(path_high)
# get_fdata applies the header scale, which is what NiiVue reads too.
highres = high_img.get_fdata().astype(np.float32)
highres_affine = np.asarray(high_img.affine, dtype=float)
high_spacing = np.linalg.norm(highres_affine[:3, :3], axis=0)
vmax = float(np.percentile(highres, 99))

print("file", path_high.name)
print("shape", highres.shape)
print("spacing mm", np.round(high_spacing, 3))
print("affine\n", np.round(highres_affine, 3))
print("vmax (99th percentile)", round(vmax, 1))


file MR_Gd.nii.gz
shape (176, 188, 144)
spacing mm [0.977 0.977 1.003]
affine
 [[ 9.76000e-01  2.20000e-02 -8.00000e-03 -8.57130e+01]
 [-2.10000e-02  9.70000e-01  1.12000e-01 -1.12453e+02]
 [ 1.00000e-02 -1.09000e-01  9.96000e-01 -5.68180e+01]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]
vmax (99th percentile) 615.8


In [ ]:
show_niivue([path_high], vmax=vmax)


## Tilted low-resolution slab

The slab keeps the in-plane field of view of `MR_Gd`. The voxels are coarser: 2.5 × 2.5 mm in plane, 7.5 mm slice thickness, 3 slices. It is tilted 25° about the source i axis, and its center sits on the center of `MR_Gd`. The in-plane size comes from the corners of `MR_Gd`, measured in the slab plane.


In [4]:
low_spacing = np.array([2.5, 2.5, 7.5])
n_slices = 3

center_index = (np.array(highres.shape[:3]) - 1) / 2.0
center_world = apply_affine(highres_affine, center_index)
tilt = directions_of(highres_affine) @ rotation_about_x(25)

ends = np.array(highres.shape[:3]) - 1.0
corner_list = [[i, j, k] for i in (0.0, ends[0]) for j in (0.0, ends[1]) for k in (0.0, ends[2])]
corners = apply_affine(highres_affine, np.array(corner_list))
in_plane = (corners - center_world) @ tilt
span_i = in_plane[:, 0].max() - in_plane[:, 0].min()
span_j = in_plane[:, 1].max() - in_plane[:, 1].min()
n_i = int(np.ceil(span_i / low_spacing[0])) + 1
n_j = int(np.ceil(span_j / low_spacing[1])) + 1
lowres_shape = (n_i, n_j, n_slices)
lowres_affine = grid_affine(lowres_shape, low_spacing, tilt, center_world)

print("lowres shape", lowres_shape)
print("in-plane span mm", np.round([span_i, span_j], 1), "slice thickness mm", low_spacing[2])
print("spacing from affine", np.round(np.linalg.norm(lowres_affine[:3, :3], axis=0), 3))


lowres shape (70, 92, 3)
in-plane span mm [170.9 226.1] slice thickness mm 7.5
spacing from affine [2.5 2.5 7.5]


## Resample onto the slab

For every slab voxel: index to millimetres with the slab affine, millimetres back to an index in `MR_Gd`, then linear interpolation. Outside the source the value is 0. That file is `MR_Gd_tilt_lowres.nii.gz`.

In NiiVue both volumes are in millimetres, so the slab lies across the brain, tilted. Switch the coordinates button to voxel indices and the grids no longer share a position: the slab is about 70×92×3 and `MR_Gd` is 176×188×144.


In [5]:
lowres = resample_to_grid(highres, highres_affine, lowres_shape, lowres_affine)
path_low = save(lowres, lowres_affine, "MR_Gd_tilt_lowres.nii.gz")
print(path_low.name, lowres.shape, "min", float(lowres.min()), "max", float(lowres.max()))


MR_Gd_tilt_lowres.nii.gz (70, 92, 3) min 0.0 max 1440.3424072265625


In [ ]:
show_niivue(
    [path_high, path_low],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.55],
    vmax=vmax,
)


## Same grid, so the two volumes can be compared

Sample `MR_Gd` onto the slab again. The arrays are now the same shape and the same affine, and the values match the slab. The checkboxes turn each copy on or off.


In [6]:
high_on_low = resample_to_grid(highres, highres_affine, lowres.shape, lowres_affine)
path_high_on_low = save(high_on_low, lowres_affine, "MR_Gd_highres_on_lowres.nii.gz")
print("max abs diff", float(np.max(np.abs(high_on_low - lowres))))


max abs diff 0.0


In [ ]:
show_niivue(
    [path_low, path_high_on_low],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.45],
    vmax=vmax,
)


## Footprint average versus nibabel

`resample_from_to` reads one interpolated value at the center of each low-resolution voxel. A 7.5 mm slice can instead average the high-resolution signal over the whole voxel: 2.5 × 2.5 mm in plane and 7.5 mm through the slice.

The footprint places several samples inside each slab voxel and averages them. Same grid, different values. The difference is the in-slice average. Turn one checkbox off to see the other image alone.


In [7]:
from nibabel.processing import resample_from_to

high_nii = nib.Nifti1Image(highres, highres_affine)
nib_low = np.asanyarray(
    resample_from_to(high_nii, (lowres_shape, lowres_affine), order=1).dataobj,
    dtype=np.float32,
)
print("center sample vs nibabel, max abs", float(np.max(np.abs(nib_low - lowres))))

samples = np.maximum(3, np.round(low_spacing / high_spacing).astype(int))
foot = resample_footprint(highres, highres_affine, lowres_shape, lowres_affine, samples)
path_foot = save(foot, lowres_affine, "MR_Gd_tilt_lowres_footprint.nii.gz")
path_nib = save(nib_low, lowres_affine, "MR_Gd_tilt_lowres_nibabel.nii.gz")
print("samples inside each voxel", tuple(int(n) for n in samples))
print("footprint vs nibabel, mean abs", float(np.mean(np.abs(foot - nib_low))))


center sample vs nibabel, max abs 0.0
samples inside each voxel (3, 3, 7)
footprint vs nibabel, mean abs 17.30442237854004


In [ ]:
show_niivue(
    [path_nib, path_foot],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.5],
    vmax=vmax,
)


## Slab back onto the high-resolution grid

Same function, grids swapped. The output has the shape of `MR_Gd` again. Only voxels that land inside the slab get intensities. Everywhere else stays 0. In NiiVue the red band is that slab, back in the original orientation.


In [8]:
back = resample_to_grid(lowres, lowres_affine, highres.shape, highres_affine)
path_back = save(back, highres_affine, "MR_Gd_lowres_on_highres.nii.gz")
print(path_back.name, back.shape)


MR_Gd_lowres_on_highres.nii.gz (176, 188, 144)


In [ ]:
show_niivue(
    [path_high, path_back],
    colormaps=["gray", "red"],
    opacities=[0.35, 1.0],
    vmax=vmax,
)


## Both volumes at 0.5 mm, in the slab frame

Same orientation and the same field of view as the slab, voxels 0.5 mm isotropic. The row from `MR_Gd` is sharper. The row from the 2.5 × 2.5 × 7.5 mm slab is the same cut, smoothed by the coarse voxels. That resample uses `mode="nearest"` so the last of the three slices is kept.


In [9]:
iso = 0.5
span = (np.asarray(lowres_shape) - 1) * low_spacing
iso_shape = tuple(int(n) for n in (np.round(span / iso).astype(int) + 1))
iso_affine = grid_affine(iso_shape, (iso, iso, iso), tilt, center_world)

high_on_iso = resample_to_grid(highres, highres_affine, iso_shape, iso_affine)
low_on_iso = resample_to_grid(lowres, lowres_affine, iso_shape, iso_affine, mode="nearest")
path_iso_high = save(high_on_iso, iso_affine, "MR_Gd_slab_0.5mm.nii.gz")
path_iso_low = save(low_on_iso, iso_affine, "MR_Gd_tilt_lowres_0.5mm.nii.gz")

low_center_k = (lowres_shape[2] - 1) / 2.0
iso_center_k = (iso_shape[2] - 1) / 2.0
ks_iso = [
    int(round(iso_center_k + (k - low_center_k) * (low_spacing[2] / iso)))
    for k in range(lowres_shape[2])
]
print("0.5 mm shape", iso_shape, "span mm", np.round(span, 1))
print("slab slices land on k", ks_iso)


0.5 mm shape (346, 456, 31) span mm [172.5 227.5  15. ]
slab slices land on k [0, 15, 30]


In [ ]:
show_niivue(
    [path_iso_high, path_iso_low],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.5],
    vmax=vmax,
)


## Other viewers

NiiVue is the viewer in this notebook because it reads the affine and draws in world millimetres.

- NiiVue also runs as a VS Code / Cursor extension, `KorbinianEckstein.niivue`. Open a `.nii.gz`, or select two and run **NiiVue: Compare**. Standalone: https://niivue.github.io/niivue-vscode/
- 3D Slicer, MITK, FSLeyes, and SPM read the affine as well.
- napari draws the data array. For a tilted grid, pass the affine yourself.

## Other resample tools

`resample_to_grid` is the short form of the same idea:

- `nibabel.processing.resample_from_to` (center sample; it matched our function exactly above)
- `nilearn.image.resample_img`
- `ants.resample_image_to_target`
- SimpleITK `Resample`

They differ in the interpolator and in what happens at the edge of the grid. A thick slice is still one sample at the voxel center unless you average over the voxel, as in the footprint section.

When the pose is unknown, resampling is not enough. The search tools are ANTs, DIPY, SimpleITK, SPM, and FSL. The next section uses ANTs and DIPY.


## Register `MR_Gd` onto the slab

The low-resolution slab is the target grid. The moving image is the high-resolution scan.

Two ways to bring it there:

1. The affine is known. A shift and an 8° rotation move `MR_Gd` off the slab. The inverse puts the matrix back, and resampling onto the slab matches the slab already built above.
2. The inverse is not given. ANTs and DIPY search for a rigid pose and resample `MR_Gd` onto the slab.


### Known affine

Same intensities, new affine. In NiiVue the green scan sits away from the gray slab. A viewer that ignores the affine would show no change, because the data array is the same.


In [10]:
center_index = (np.array(highres.shape) - 1) / 2.0
center_world = apply_affine(highres_affine, center_index)
shift_mm = np.array([12.0, -8.0, 6.0])
R = rotation_about_x(8)
world_rigid = np.eye(4)
world_rigid[:3, :3] = R
world_rigid[:3, 3] = center_world + shift_mm - R @ center_world
moved_affine = world_rigid @ highres_affine
path_moved = save(highres, moved_affine, "MR_Gd_moved_affine.nii.gz")
print("shift mm", shift_mm, "rotation deg", 8)


shift mm [12. -8.  6.] rotation deg 8


In [ ]:
show_niivue(
    [path_low, path_moved],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.55],
    vmax=vmax,
)


The inverse of that rigid transform restores the original affine. Resampling onto the slab is then the same operation as the first resample, so the values match the slab.


In [11]:
corrected_affine = np.linalg.inv(world_rigid) @ moved_affine
on_slab_known = resample_to_grid(highres, corrected_affine, lowres.shape, lowres_affine)
path_known = save(on_slab_known, lowres_affine, "MR_Gd_on_slab_known.nii.gz")
print("affine restored, max abs", float(np.max(np.abs(corrected_affine - highres_affine))))
print("on slab vs slab, max abs", float(np.max(np.abs(on_slab_known - lowres))))


affine restored, max abs 1.4210854715202004e-14
on slab vs slab, max abs 0.0


In [ ]:
show_niivue(
    [path_low, path_known],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.5],
    vmax=vmax,
)


### ANTs rigid

ANTs is not given the inverse. It estimates a rigid transform from the moved high-resolution scan to the slab, then resamples onto the slab grid. Three thick slices make the metric a little soft, so the result sits close to the slab without matching every voxel.


In [12]:
import ants

ants_fixed = ants.image_read(str(path_low))
ants_moving = ants.image_read(str(path_moved))
reg = ants.registration(fixed=ants_fixed, moving=ants_moving, type_of_transform="Rigid")
warped = ants.apply_transforms(
    fixed=ants_fixed, moving=ants_moving, transformlist=reg["fwdtransforms"]
)
path_ants = OUT / "MR_Gd_on_slab_ants.nii.gz"
ants.image_write(warped, str(path_ants))
ants_data = nib.load(path_ants).get_fdata().astype(np.float32)
print("ants vs slab, correlation", float(np.corrcoef(lowres.ravel(), ants_data.ravel())[0, 1]))
print("ants vs slab, mean abs", float(np.mean(np.abs(ants_data - lowres))))


ants vs slab, correlation 0.9004255300663
ants vs slab, mean abs 48.234344482421875


In [ ]:
show_niivue(
    [path_low, path_ants],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.5],
    vmax=vmax,
)


### DIPY rigid

Same pair. DIPY searches a rigid transform with mutual information and resamples the moved scan onto the slab.


In [13]:
from dipy.align.imaffine import AffineRegistration, MutualInformationMetric
from dipy.align.transforms import RigidTransform3D

dipy_reg = AffineRegistration(
    metric=MutualInformationMetric(nbins=32, sampling_proportion=None),
    level_iters=[100, 100, 50],
    sigmas=[3.0, 1.0, 0.0],
    factors=[4, 2, 1],
)
rigid = dipy_reg.optimize(
    static=lowres,
    moving=highres,
    transform=RigidTransform3D(),
    params0=None,
    static_grid2world=lowres_affine,
    moving_grid2world=moved_affine,
)
dipy_data = np.asanyarray(rigid.transform(highres), dtype=np.float32)
path_dipy = save(dipy_data, lowres_affine, "MR_Gd_on_slab_dipy.nii.gz")
print("dipy vs slab, correlation", float(np.corrcoef(lowres.ravel(), dipy_data.ravel())[0, 1]))
print("dipy vs slab, mean abs", float(np.mean(np.abs(dipy_data - lowres))))


[2026-10-02 12:10:58][dipy] INFO: Optimizing level 2 [max iter: 100]


[2026-10-02 12:10:58][dipy] INFO: Optimizing level 1 [max iter: 100]


[2026-10-02 12:10:59][dipy] INFO: Optimizing level 0 [max iter: 50]


dipy vs slab, correlation 0.9997807155735023
dipy vs slab, mean abs 2.0170271396636963


In [ ]:
show_niivue(
    [path_low, path_dipy],
    colormaps=["gray", "green"],
    opacities=[1.0, 0.5],
    vmax=vmax,
)
